# 12 — Capstone: Full Research Run

Everything together. `ResearchEngine.run(topic)` orchestrates the full pipeline:

1. Build agents with bridge observers for reactive coordination
2. Run root team (6 specialists via handoff routing)
3. Reactive depth expansion (novelty-gated child teams)
4. Wait for tree to quiesce
5. Cross-check all findings
6. Iterative paper loop with gap feedback
7. Return `PaperDraft` with quality score

Set `OPENAI_API_KEY` and `EXA_API_KEY` in your `.env`. Optionally set `KHIVE_API_KEY` for persistent knowledge.

In [ ]:
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

## Configuration

| Parameter | Default | Description |
|---|---|---|
| `model` | `gpt-5.4-mini` | Model for all agents (1M context) |
| `max_depth` | 3 | Maximum recursion depth |
| `max_concurrent` | 5 | Max concurrent depth nodes |
| `novelty_threshold` | 0.7 | Minimum novelty to spawn a child |
| `paper_max_iterations` | 2 | Max paper rewrite cycles |
| `paper_quality_threshold` | 0.7 | Quality score to accept paper |
| `khive_api_key` | env | Optional khive for persistent knowledge |
| `on_event` | None | SSE callback for live progress |

In [ ]:
from lionag2 import ResearchEngine

events: list[dict] = []


def on_event(e: dict) -> None:
    events.append(e)
    t = e.get("type", "?")
    detail = e.get("claim", e.get("agent", e.get("question", "")))
    if t in (
        "tree_init",
        "child_spawned",
        "node_complete",
        "exploration_done",
        "cross_check_done",
        "paper_draft",
        "finding",
    ):
        msg = f"  [{t}]" + (f" {str(detail)[:60]}" if detail else "")
        print(msg)


engine = ResearchEngine(
    model="gpt-5.4-mini",
    max_depth=1,  # keep it fast for demo
    novelty_threshold=0.85,  # only very novel findings spawn children
    paper_max_iterations=1,  # single paper pass
    on_event=on_event,
)

## Run

This will take a few minutes. You'll see live events as the team works.

In [ ]:
paper = await engine.run("What are the failure modes of chain-of-thought prompting in LLMs?")

print(f"\nQuality: {paper.quality_score:.2f}")
print(f"Length: {len(paper.body_markdown):,} chars")
print(f"Gaps remaining: {len(paper.gaps)}")
print(f"Limitations: {len(paper.limitations)}")

## Render the paper

In [ ]:
display(Markdown(paper.as_markdown()))

## Event stats

In [ ]:
from collections import Counter

counts = Counter(e.get("type") for e in events)
print(f"Total events: {len(events)}")
for event_type, count in counts.most_common():
    print(f"  {event_type}: {count}")

# Findings summary
findings = [e for e in events if e.get("type") == "finding"]
print(f"\nFindings: {len(findings)}")
for f in findings:
    print(
        f"  [{f.get('source')} d={f.get('depth')}] "
        f"novelty={f.get('novelty', 0):.2f} — {f.get('claim', '')[:60]}"
    )

## Captured URLs

URL capture is handled by observers on each agent — `ToolResultsEvent` observer extracts `title → url` from Exa results and records them as `UrlCaptured` events in the Flow.

In [ ]:
from lionag2.research.events import UrlCaptured

urls = engine.flow.items.by_type(UrlCaptured)
print(f"Captured {len(urls)} URLs:")
for u in list(urls)[:10]:
    print(f"  [{u.title[:50]}]({u.url})")

## What we built

| Component | AG2 primitive | lionag2 layer |
|---|---|---|
| Agents | `Agent` + `OpenAIConfig` | 7 specialist prompts + tool mapping |
| Context | `ConversationPolicy` + `SlidingWindowPolicy` | `SafeSlidingWindowPolicy` (orphan-safe) |
| Coordination | `@agent.observer` (sync or async) | Bridge observers for typed events |
| Routing | `handoff()` tool | Handoff-based agent-to-agent routing |
| Events | `BaseEvent` subclasses | 8 typed research + coordination events |
| Tools | `@tool` decorator | Emission + handoff tools |
| State | `MemoryStream` | `Flow` — shared Pile + named Progressions |
| Middleware | `ToolMiddleware` | HTML cleaning for Exa results |
| Structured output | `response_schema` (native) | `CrossCheckReport`, `PaperDraft` |
| Serving | `AGUIStream` | Starlette ASGI app (AG2 ag-ui compatible) |

Total: pure AG2 beta primitives. No external orchestration framework. The pattern transfers to any recursive multi-agent application by changing the schemas, prompts, and tools.